# Validation File Comparison

Compares agreement between two validation CSV files on a per-ID basis after normalizing both to a common boolean representation.

In [34]:
import pandas as pd
from pathlib import Path
from statsmodels.stats.contingency_tables import mcnemar

## Configuration

In [35]:
VALIDATION_DIR = Path(".")  # relative to this notebook

FILE_A = VALIDATION_DIR / "core_bench_easy_ans_form_multi.csv"
FILE_B = VALIDATION_DIR / "core_bench_easy_oh1_3pointscale.csv"

## Conversion Functions

Both files are normalized to a common boolean `value` column:

- **`ans_form_multi`** format: `label_oh1` is `True` (string) or blank → `True` if set, `False` if blank
- **`oh1_3pointscale`** format: `target` is a numeric score (0, 1, 2) → `False` if 0, `True` otherwise

In [36]:
def load_ans_form_multi(path: Path) -> pd.DataFrame:
    """Load ans_form_multi format. Converts label_oh1 to boolean value."""
    df = pd.read_csv(path)
    df["value"] = df["label_oh1"].fillna("").astype(str).str.strip().str.lower() == "true"
    return df[["id", "value"]]


def load_oh1_3pointscale(path: Path) -> pd.DataFrame:
    """Load oh1_3pointscale format. Converts target to boolean: 0 → False, else → True."""
    df = pd.read_csv(path)
    df["value"] = df["target"] != 0
    return df[["id", "value"]]

## Load & Inspect

In [37]:
df_a = load_ans_form_multi(FILE_A)
df_b = load_oh1_3pointscale(FILE_B)

print(f"File A ({FILE_A.name}): {len(df_a)} rows")
print(f"  True: {df_a['value'].sum()}, False: {(~df_a['value']).sum()}")
df_a.head()

File A (core_bench_easy_ans_form_multi.csv): 45 rows
  True: 26, False: 19


,id,value
0,2HMRtkCxgju7q87nBz5Msm,True
1,2grUTB3MXmZJW3QcxUKygu,False
2,4LWaZXamgv9M6GVrA78zgv,True
3,5QHujhrnsNuSEfThXSukZi,False
4,86R7tfNjB4UxNEbT2FXva2,True


In [38]:
print(f"File B ({FILE_B.name}): {len(df_b)} rows")
print(f"  True: {df_b['value'].sum()}, False: {(~df_b['value']).sum()}")
df_b.head()

File B (core_bench_easy_oh1_3pointscale.csv): 45 rows
  True: 27, False: 18


,id,value
0,2HMRtkCxgju7q87nBz5Msm,True
1,4LWaZXamgv9M6GVrA78zgv,True
2,5QHujhrnsNuSEfThXSukZi,True
3,AGLCdV4EYk534irmHdooSr,True
4,CqFpbjm5SccfTZrJjHkVvz,True


## Merge & Compare

In [39]:
merged = df_a.merge(df_b, on="id", how="outer", suffixes=("_a", "_b"), indicator=True)

only_in_a = merged[merged["_merge"] == "left_only"]["id"].tolist()
only_in_b = merged[merged["_merge"] == "right_only"]["id"].tolist()

if only_in_a:
    print(f"IDs only in File A ({len(only_in_a)}): {only_in_a}")
if only_in_b:
    print(f"IDs only in File B ({len(only_in_b)}): {only_in_b}")

common = merged[merged["_merge"] == "both"].copy()
print(f"\nIDs in common: {len(common)}")


IDs in common: 45


In [40]:
common["agree"] = common["value_a"] == common["value_b"]

n_agree = common["agree"].sum()
n_disagree = (~common["agree"]).sum()
pct = 100 * n_agree / len(common)

print(f"Agreement:    {n_agree}/{len(common)} ({pct:.1f}%)")
print(f"Disagreement: {n_disagree}/{len(common)} ({100 - pct:.1f}%)")

Agreement:    44/45 (97.8%)
Disagreement: 1/45 (2.2%)


## Disagreements

In [41]:
disagreements = common[~common["agree"]][["id", "value_a", "value_b"]].copy()
disagreements.columns = ["id", FILE_A.stem, FILE_B.stem]
disagreements.reset_index(drop=True, inplace=True)
disagreements

,id,core_bench_easy_ans_form_multi,core_bench_easy_oh1_3pointscale
0,5QHujhrnsNuSEfThXSukZi,False,True


## McNemar's Test

Tests whether the two raters disagree significantly. McNemar's test uses only the off-diagonal counts (discordant pairs):
- **b**: A=True, B=False
- **c**: A=False, B=True

H₀: the marginal distributions are equal (no systematic bias between raters).

The exact binomial version is used when b+c < 25, otherwise the chi-squared approximation.

In [42]:
# Contingency table counts
tt = ((common["value_a"] == True)  & (common["value_b"] == True)).sum()   # a
tf = ((common["value_a"] == True)  & (common["value_b"] == False)).sum()  # b
ft = ((common["value_a"] == False) & (common["value_b"] == True)).sum()   # c
ff = ((common["value_a"] == False) & (common["value_b"] == False)).sum()  # d

table = [[tt, tf], [ft, ff]]

print("Contingency table (rows=File A, cols=File B):")
print(f"              B=True  B=False")
print(f"  A=True      {tt:5d}   {tf:5d}")
print(f"  A=False     {ft:5d}   {ff:5d}")
print(f"\nDiscordant pairs (b+c): {tf + ft}")

# Use exact test when discordant pairs are small
exact = (tf + ft) < 25
result = mcnemar(table, exact=exact)

print(f"\nMcNemar's test ({'exact binomial' if exact else 'chi-squared'}):")
print(f"  statistic = {result.statistic:.4f}")
print(f"  p-value   = {result.pvalue:.4f}")

if result.pvalue < 0.05:
    print("\n  → Significant difference between raters (p < 0.05)")
else:
    print("\n  → No significant difference between raters (p ≥ 0.05)")

Contingency table (rows=File A, cols=File B):
              B=True  B=False
  A=True         26       0
  A=False         1      18

Discordant pairs (b+c): 1

McNemar's test (exact binomial):
  statistic = 0.0000
  p-value   = 1.0000

  → No significant difference between raters (p ≥ 0.05)


In [43]:
from sklearn.metrics import cohen_kappa_score

kappa = cohen_kappa_score(common["value_a"], common["value_b"])

def kappa_interpretation(k):
    if k < 0.20: return "slight"
    if k < 0.40: return "fair"
    if k < 0.60: return "moderate"
    if k < 0.80: return "substantial"
    return "almost perfect"

print(f"Cohen's κ = {kappa:.4f}  ({kappa_interpretation(kappa)} agreement)")

Cohen's κ = 0.9541  (almost perfect agreement)


## Full Comparison Table

In [44]:
display_df = common[["id", "value_a", "value_b", "agree"]].copy()
display_df.columns = ["id", FILE_A.stem, FILE_B.stem, "agree"]
display_df = display_df.sort_values("agree").reset_index(drop=True)
display_df

,id,core_bench_easy_ans_form_multi,core_bench_easy_oh1_3pointscale,agree
0,5QHujhrnsNuSEfThXSukZi,False,True,False
1,2HMRtkCxgju7q87nBz5Msm,True,True,True
2,YeMcaRCoKeqcsyhA3sgXfE,True,True,True
3,Zet6yJtmDinpesbV5oJj5w,False,False,True
4,ZgpNUY6Vkio7HjQTGcTqCr,False,False,True
5,ZvcecK5MsMs2QqSqrXrGP7,True,True,True
6,Zwpz7JZWhncQhG6c8kCP33,False,False,True
7,afiDxGmiZywARrat8YRiWr,True,True,True
8,dERThuzybMVzSxn2cK4UEa,True,True,True
9,dGSdhwiATC2tiF7pg2oR79,False,False,True
